# Manga / Webtoon → 대사 추출 + 번역 (Colab)

`PDF/이미지 → RF-DETR → OCR → 언어 확인 → 외국어만 번역 → JSONL/TXT`

단일 이미지 / 여러 이미지 / PDF를 자동 판별하고, 실행 단계마다 진단 로그를 출력합니다.

## 1. 패키지 설치
PaddlePaddle은 CPU oneDNN 오류를 피하려고 3.2.2로 고정하며, Colab T4 기준 GPU 버전을 설치합니다.


In [ ]:
!pip uninstall -y -q paddlepaddle paddlepaddle-gpu paddleocr paddlex
!pip install -q paddlepaddle-gpu==3.2.2 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!pip install -q paddleocr==3.3.2 paddlex==3.3.13 rfdetr==1.7.0 safetensors huggingface_hub manga-ocr transformers accelerate bitsandbytes lingua-language-detector pymupdf pillow numpy tqdm matplotlib
!rm -rf /content/manga2text_tmp
!git clone -q https://github.com/HisameOgasahara/manga2text_tmp.git /content/manga2text_tmp
print('[설치 완료]')

## 2. 실행 환경 확인
오류가 나면 이 셀의 출력부터 확인하면 됩니다.


In [ ]:
import platform, torch, paddle, paddleocr, paddlex
print('[실행 환경]')
print('  Python       :', platform.python_version())
print('  PyTorch      :', torch.__version__)
print('  PyTorch CUDA :', torch.cuda.is_available())
if torch.cuda.is_available(): print('  GPU          :', torch.cuda.get_device_name(0))
print('  Paddle       :', paddle.__version__)
print('  PaddleOCR    :', paddleocr.__version__)
print('  PaddleX      :', paddlex.__version__)
print('  Paddle CUDA  :', paddle.device.is_compiled_with_cuda())
print('  Paddle 장치  :', paddle.device.get_device())

## 3. 사용자 설정

아래 폼만 바꾸면 됩니다. Colab에서는 GPU 사용을 기본값으로 두었습니다.

In [ ]:
import sys
from pathlib import Path
sys.path.append('/content/manga2text_tmp')
from manga2text_pipeline import auto_detect_source_language, build_language_detector, classify_inputs, collect_page_images, describe_input_mode, load_koharu_detector, load_ocr_backend, load_translation_model, make_preview_images, process_pages, resolve_ocr_configuration, save_results

언어_자동_판별 = True # @param {type:"boolean"}
원문_언어 = '한국어' # @param ['한국어', '일본어', '중국어', '영어']
글자_읽는_방법 = '자동' # @param ['자동', 'MangaOCR', 'PaddleOCR']
읽기_방향 = '자동' # @param ['자동', '오른쪽→왼쪽 (일본 만화)', '왼쪽→오른쪽 (웹툰/영문)']
PaddleOCR_계산_장치 = 'GPU' # @param ['GPU', 'CPU']
외국어_한국어_번역 = True # @param {type:"boolean"}
번역_모델 = 'Qwen3-1.7B (가볍고 빠름)' # @param ['Qwen3-1.7B (가볍고 빠름)', 'Qwen3-4B (품질 우선)']
효과음도_읽기 = False # @param {type:"boolean"}
PDF_화질_DPI = 200 # @param {type:"integer"}
처리할_페이지_수 = 0 # @param {type:"integer"}
동시에_준비할_작업_수 = 4 # @param {type:"integer"}
진단_로그_보기 = True # @param {type:"boolean"}

LANG={'한국어':'ko','일본어':'ja','중국어':'zh','영어':'en'}
OCR={'자동':'auto','MangaOCR':'manga','PaddleOCR':'paddle'}
DIR={'자동':'auto','오른쪽→왼쪽 (일본 만화)':'rtl','왼쪽→오른쪽 (웹툰/영문)':'ltr'}
MODEL={'Qwen3-1.7B (가볍고 빠름)':'Qwen/Qwen3-1.7B','Qwen3-4B (품질 우선)':'Qwen/Qwen3-4B'}
AUTO_DETECT_SOURCE_LANGUAGE=언어_자동_판별
SOURCE_LANGUAGE=LANG[원문_언어]
OCR_BACKEND=OCR[글자_읽는_방법]
READING_DIRECTION=DIR[읽기_방향]
PADDLE_DEVICE='gpu' if PaddleOCR_계산_장치=='GPU' else 'cpu'
ENABLE_TRANSLATION=외국어_한국어_번역
TRANSLATION_MODEL=MODEL[번역_모델]
INCLUDE_SFX=효과음도_읽기
PDF_DPI=PDF_화질_DPI
PAGE_LIMIT=None if 처리할_페이지_수<=0 else 처리할_페이지_수
INPUT_WORKERS=max(1,동시에_준비할_작업_수)
DEBUG_LOG=진단_로그_보기
MAX_NEW_TOKENS=256; CROP_PADDING=8; ROW_TOLERANCE=80; AUTO_LANGUAGE_SAMPLE_CROPS=3; DEBUG_SAMPLES_PER_PAGE=3
CLASS_THRESHOLDS={0:0.25,1:0.20,2:0.50,3:0.50}
WORK_DIR=Path('/content/manga2text'); INPUT_DIR=WORK_DIR/'input'; PAGE_DIR=WORK_DIR/'pages'; OUTPUT_DIR=WORK_DIR/'output'
for d in [INPUT_DIR,PAGE_DIR,OUTPUT_DIR]: d.mkdir(parents=True,exist_ok=True)
print('[현재 설정]')
print('  언어 자동 판별      :', AUTO_DETECT_SOURCE_LANGUAGE)
print('  원문 언어            :', 원문_언어)
print('  글자 읽는 방법       :', 글자_읽는_방법)
print('  PaddleOCR 계산 장치  :', PaddleOCR_계산_장치)
print('  외국어 번역          :', ENABLE_TRANSLATION)
if PADDLE_DEVICE=='gpu' and not paddle.device.is_compiled_with_cuda(): print('[경고] GPU를 골랐지만 Paddle GPU가 잡히지 않았습니다.')

## 4. 만화 업로드 + 자동 판별 + 썸네일


In [ ]:
import shutil, matplotlib.pyplot as plt
from google.colab import files
if INPUT_DIR.exists(): shutil.rmtree(INPUT_DIR)
INPUT_DIR.mkdir(parents=True,exist_ok=True)
uploaded=files.upload()
for name,data in uploaded.items(): (INPUT_DIR/name).write_bytes(data)
groups=classify_inputs(INPUT_DIR); mode=describe_input_mode(groups)
print('[입력 확인]')
print('  입력 종류 :',mode); print('  이미지 수 :',len(groups['images'])); print('  PDF 수    :',len(groups['pdfs']))
previews=make_preview_images(INPUT_DIR,max_items=8,pdf_preview_pages=3)
if not previews: raise RuntimeError('처리할 이미지/PDF가 없습니다.')
cols=min(4,len(previews)); rows=(len(previews)+cols-1)//cols
plt.figure(figsize=(4*cols,5*rows))
for i,(label,img) in enumerate(previews,1):
    plt.subplot(rows,cols,i); plt.imshow(img); plt.title(label); plt.axis('off')
plt.tight_layout(); plt.show()

## 5. 페이지 이미지 준비
PDF 페이지 변환과 여러 이미지 준비는 CPU에서 병렬 처리합니다.


In [ ]:
if PAGE_DIR.exists(): shutil.rmtree(PAGE_DIR)
PAGE_DIR.mkdir(parents=True,exist_ok=True)
print('[페이지 준비] 작업 수=',INPUT_WORKERS,' DPI=',PDF_DPI,' 제한=',PAGE_LIMIT or '전체')
page_paths=collect_page_images(INPUT_DIR,PAGE_DIR,pdf_dpi=PDF_DPI,page_limit=PAGE_LIMIT,workers=INPUT_WORKERS)
print('  준비된 페이지:',len(page_paths))
if not page_paths: raise RuntimeError('처리할 페이지가 없습니다.')

## 6. 대사 위치 찾기 모델(RF-DETR)
글자를 읽기 전에 대사가 있는 위치부터 찾습니다.


In [ ]:
print('[대사 위치 모델 로드]')
detector=load_koharu_detector()
print('  CUDA:',torch.cuda.is_available())
if torch.cuda.is_available(): print('  GPU :',torch.cuda.get_device_name(0))

## 7. 원문 언어와 OCR 방식 결정
자동 판별이면 앞부분 대사를 여러 OCR로 시험하고, 샘플 문자열과 점수를 출력합니다.


In [ ]:
if AUTO_DETECT_SOURCE_LANGUAGE:
    selected_source_language,details=auto_detect_source_language(page_paths,detector,CLASS_THRESHOLDS,crop_padding=CROP_PADDING,max_crops=AUTO_LANGUAGE_SAMPLE_CROPS,paddle_device=PADDLE_DEVICE)
else:
    selected_source_language=SOURCE_LANGUAGE; print('[언어 수동 선택]',원문_언어)
ocr_config=resolve_ocr_configuration(selected_source_language,OCR_BACKEND,READING_DIRECTION)
print('[결정된 처리 방법]')
print('  원문 언어      :',ocr_config['source_language'])
print('  사용할 OCR     :',ocr_config['ocr_backend'])
print('  PaddleOCR 언어 :',ocr_config['paddle_lang'])
print('  읽기 방향      :',ocr_config['reading_direction'])
print('  계산 장치      :',PADDLE_DEVICE if ocr_config['ocr_backend']=='paddle' else 'MangaOCR')

## 8. OCR 모델 로드


In [ ]:
ocr_model=load_ocr_backend(ocr_config['ocr_backend'],ocr_config['paddle_lang'],PADDLE_DEVICE)
print('[OCR 모델 로드 완료]')

## 9. 언어 확인 + 번역 모델 로드
한국어로 읽힌 대사는 번역하지 않습니다.


In [ ]:
language_detector,language_to_code=build_language_detector()
translation_tokenizer=None; translation_model=None
if ENABLE_TRANSLATION:
    translation_tokenizer,translation_model=load_translation_model(TRANSLATION_MODEL)
    print('[번역 모델 로드 완료]',TRANSLATION_MODEL)
else: print('[번역 사용 안 함]')

## 10. 전체 처리 실행
진단 로그에는 페이지별 검출 수, OCR 샘플, 언어 판정, 번역 여부가 표시됩니다.


In [ ]:
records=process_pages(page_paths=page_paths,detector=detector,ocr_backend=ocr_config['ocr_backend'],ocr_model=ocr_model,language_detector=language_detector,language_to_code=language_to_code,class_thresholds=CLASS_THRESHOLDS,reading_direction=ocr_config['reading_direction'],row_tolerance=ROW_TOLERANCE,crop_padding=CROP_PADDING,include_sfx=INCLUDE_SFX,enable_translation=ENABLE_TRANSLATION,translation_tokenizer=translation_tokenizer,translation_model=translation_model,max_new_tokens=MAX_NEW_TOKENS,debug=DEBUG_LOG,debug_samples_per_page=DEBUG_SAMPLES_PER_PAGE)
print('[전체 처리 완료] 추출 대사:',len(records))

## 11. 결과 미리보기


In [ ]:
for r in records[:30]:
    print(f"[p.{r['page']:03d}/{r['order']:02d}] {r['language']} | {r['original']} -> {r['korean']}")

## 12. JSONL / TXT 저장 + 다운로드


In [ ]:
jsonl_path,txt_path=save_results(records,OUTPUT_DIR)
print('[저장 완료]',jsonl_path,txt_path,sep='\n  ')
files.download(str(jsonl_path)); files.download(str(txt_path))